# Ollama Installer
> Install MCP configuration for Ollama

In [ ]:
#| default_exp install.ollama

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from typing import Optional

from nbdev_mcp.install.base import (
    BaseInstaller, InstallResult, MCPServerConfig,
    read_json_config, write_json_config, get_python_path, which,
    get_ollama_config_path
)

## Ollama Installer

In [ ]:
#| export
class OllamaInstaller(BaseInstaller):
    """Installer for Ollama.
    
    Configures MCP in Ollama's config.json file.
    Location varies by platform:
    - macOS: ~/.ollama/config.json
    - Windows: %LOCALAPPDATA%/Ollama/config.json
    - Linux: ~/.ollama/config.json
    
    Note: MCP support in Ollama may require specific versions.
    """
    
    @property
    def name(self) -> str:
        return 'ollama'
    
    @property
    def config_path(self) -> Path:
        return get_ollama_config_path()
    
    def is_configured(self) -> bool:
        """Check if nbdev is already configured."""
        config = read_json_config(self.config_path)
        return 'nbdev' in config.get('mcp_servers', {})
    
    def do_install(self, server: MCPServerConfig) -> None:
        """Perform the installation."""
        config = read_json_config(self.config_path)
        
        if 'mcp_servers' not in config:
            config['mcp_servers'] = {}
        
        config['mcp_servers'][server.name] = {
            'command': server.command,
            'args': server.args,
            'transport': server.transport
        }
        write_json_config(self.config_path, config)
    
    def do_uninstall(self) -> None:
        """Remove the configuration."""
        config = read_json_config(self.config_path)
        
        if 'mcp_servers' in config and 'nbdev' in config['mcp_servers']:
            del config['mcp_servers']['nbdev']
        
        write_json_config(self.config_path, config)
    
    def is_ollama_installed(self) -> bool:
        """Check if Ollama CLI is available."""
        return which('ollama') is not None

## Convenience Functions

In [ ]:
#| export
def install_ollama(
    python_path: Optional[str] = None,
    force: bool = False,
    dry_run: bool = False
) -> InstallResult:
    """Install nbdev-mcp configuration for Ollama.
    
    Parameters
    ----------
    python_path : str, optional
        Python interpreter path. Defaults to current.
    force : bool
        Overwrite existing configuration.
    dry_run : bool
        Show what would be done without making changes.
    
    Returns
    -------
    InstallResult
        Result of the installation.
    """
    installer = OllamaInstaller()
    server = MCPServerConfig.for_nbdev(python_path)
    return installer.install(server, force=force, dry_run=dry_run)


def uninstall_ollama(dry_run: bool = False) -> InstallResult:
    """Remove nbdev-mcp configuration from Ollama."""
    installer = OllamaInstaller()
    return installer.uninstall(dry_run=dry_run)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()